# Five-fold cross-validation — BraTS 2020 activation-function ablation

The multi-seed notebook tests robustness to the random **seed**. This notebook tests robustness to the **data split**: it re-pools all **369** patient volumes and evaluates four activation functions with **5-fold cross-validation**.

| Setting | Value |
|---|---|
| Folds | **5** (`KFold`, shuffle, random_state=42) |
| Activations | **ReLU, Swish, PReLU, TanhExp** (matching the multi-seed study) |
| Seed | **fixed at 42** for every fold — only the split varies |
| Trainings | 4 activations × 5 folds = **20 runs**, 100 epochs each |
| Pipeline | identical to the main study (model, loss, optimiser, preprocessing) |

No re-preprocessing: the notebook reuses the existing 128³ `.npy` files from both the `train/` and `val/` folders (all 369 patients) and re-splits them into folds. Each file is one patient, so folds are patient-level by construction. Across the five folds every patient is held out exactly once (out-of-fold prediction).

Resumable: re-run the training cell after any disconnect. Outputs are the CSV files archived in `results/crossval/`.

## 1 · Mount and configure

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=True)

DATA_DIR     = "/content/drive/MyDrive/BraTS2020_Preprocessed_128"
RESULTS_ROOT = "/content/drive/MyDrive/BraTS_CV_Results"     # new folder; originals untouched
os.makedirs(RESULTS_ROOT, exist_ok=True)

KFOLDS      = 5
ACTIVATIONS = ["relu", "swish", "prelu", "tanhexp"]
SEED        = 42
EPOCHS      = 100
BATCH_SIZE  = 2
LR          = 3e-4

assert os.path.exists(DATA_DIR), f"Preprocessed data not found at {DATA_DIR}."
print(f"Plan: {len(ACTIVATIONS)} activations x {KFOLDS} folds = {len(ACTIVATIONS)*KFOLDS} trainings")

## 2 · Components (model, activations, dataset, seeding)
Identical to the main pipeline; the dataset takes an explicit file list so that each fold trains on its own subset.

In [ ]:
import time, random, glob
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.optim as optim, torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import KFold
from tqdm.auto import tqdm

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
def _worker_init_fn(_):
    s = torch.initial_seed() % (2**32); np.random.seed(s); random.seed(s)

class Mish(nn.Module):
    def forward(self, x): return x * torch.tanh(F.softplus(x))
def get_activation(name):
    name=name.lower()
    if name=='relu': return nn.ReLU(inplace=True)
    if name=='swish': return nn.SiLU(inplace=True)
    if name=='prelu': return nn.PReLU()
    if name=='tanhexp': return TanhExp()
    if name=='mish': return Mish()
    raise ValueError(name)
class TanhExp(nn.Module):
    def forward(self, x): return x * torch.tanh(torch.exp(torch.clamp(x, max=20)))

class ResidualBlock(nn.Module):
    def __init__(self, i, o, act):
        super().__init__()
        self.conv1=nn.Conv3d(i,o,3,padding=1); self.bn1=nn.BatchNorm3d(o); self.act=get_activation(act)
        self.conv2=nn.Conv3d(o,o,3,padding=1); self.bn2=nn.BatchNorm3d(o)
        self.skip=nn.Conv3d(i,o,1) if i!=o else nn.Identity()
    def forward(self,x): return self.act(self.bn2(self.conv2(self.act(self.bn1(self.conv1(x)))))+self.skip(x))
class ImprovedUNet3D(nn.Module):
    def __init__(self, ic, oc, act):
        super().__init__()
        self.enc1=ResidualBlock(ic,32,act); self.pool=nn.MaxPool3d(2)
        self.enc2=ResidualBlock(32,64,act); self.enc3=ResidualBlock(64,128,act); self.enc4=ResidualBlock(128,256,act)
        self.bottleneck=ResidualBlock(256,512,act)
        self.up4=nn.ConvTranspose3d(512,256,2,2); self.dec4=ResidualBlock(512,256,act)
        self.up3=nn.ConvTranspose3d(256,128,2,2); self.dec3=ResidualBlock(256,128,act)
        self.up2=nn.ConvTranspose3d(128,64,2,2);  self.dec2=ResidualBlock(128,64,act)
        self.up1=nn.ConvTranspose3d(64,32,2,2);   self.dec1=ResidualBlock(64,32,act)
        self.out=nn.Conv3d(32,oc,1)
    def forward(self,x):
        e1=self.enc1(x); e2=self.enc2(self.pool(e1)); e3=self.enc3(self.pool(e2)); e4=self.enc4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        d4=self.dec4(torch.cat([self.up4(b),e4],1)); d3=self.dec3(torch.cat([self.up3(d4),e3],1))
        d2=self.dec2(torch.cat([self.up2(d3),e2],1)); d1=self.dec1(torch.cat([self.up1(d2),e1],1))
        return self.out(d1)

def mask_for(img_path): return img_path.replace('/images/','/masks/')
class CVDataset(Dataset):
    def __init__(self, img_files): self.files=list(img_files)
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        ip=self.files[idx]; mp=mask_for(ip)
        img=np.load(ip).astype(np.float32); mask=np.load(mp).astype(np.longlong)
        return (torch.from_numpy(img).permute(3,2,0,1), torch.from_numpy(mask).permute(2,0,1),
                os.path.basename(ip))

device='cuda' if torch.cuda.is_available() else 'cpu'
print("Components ready. Device:", device)

## 3 · Pool all 369 patients and build the 5 folds

In [ ]:
all_imgs = sorted(glob.glob(f"{DATA_DIR}/train/images/*.npy") + glob.glob(f"{DATA_DIR}/val/images/*.npy"))
print(f"Pooled {len(all_imgs)} patient volumes.")
kf = KFold(n_splits=KFOLDS, shuffle=True, random_state=SEED)
FOLDS = list(kf.split(all_imgs))   # deterministic given sorted all_imgs + random_state
for k,(tr,va) in enumerate(FOLDS):
    print(f"  fold {k}: train={len(tr)}  val={len(va)}")

## 4 · Training (one run per activation × fold), resumable
Seed fixed at 42 for every run; combined unweighted cross-entropy + soft-Dice loss; AdamW + cosine annealing; AMP. Best epoch per fold selected on that fold's held-out set.

In [ ]:
def train_one_fold(act, fold, train_files, val_files, epochs=EPOCHS):
    run=f"{act}_fold{fold}"; sd=os.path.join(RESULTS_ROOT, run); os.makedirs(sd, exist_ok=True)
    ckpt=os.path.join(sd,"latest.pth"); seed_everything(SEED)
    model=ImprovedUNet3D(4,4,act).to(device); opt=optim.AdamW(model.parameters(),lr=LR)
    scaler=torch.amp.GradScaler('cuda'); ce=nn.CrossEntropyLoss(); sch=CosineAnnealingLR(opt,T_max=epochs)
    start,hist=0,[]
    if os.path.exists(ckpt):
        ck=torch.load(ckpt,map_location=device,weights_only=False)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); start=ck['epoch']; hist=ck.get('history',[])
        if 'scheduler' in ck: sch.load_state_dict(ck['scheduler'])
        else:
            for _ in range(start): sch.step()
        if start>=epochs: print(f"   {run} done."); return
        print(f"   resuming {run} @ {start}")
    g=torch.Generator(); g.manual_seed(SEED)
    tdl=DataLoader(CVDataset(train_files),batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=True,
                   generator=g,worker_init_fn=_worker_init_fn)
    vdl=DataLoader(CVDataset(val_files),batch_size=BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=True)
    for ep in range(start,epochs):
        torch.cuda.reset_peak_memory_stats(); t0=time.time(); model.train(); tl=0
        for x,y,_ in tqdm(tdl,desc=f"{run} ep{ep+1} train",leave=False):
            x,y=x.to(device),y.to(device); opt.zero_grad()
            with torch.amp.autocast('cuda'):
                p=model(x); ps=F.softmax(p,1); yo=F.one_hot(y,4).permute(0,4,1,2,3).float()
                inter=(ps*yo).sum((2,3,4)); union=ps.sum((2,3,4))+yo.sum((2,3,4))
                dl=1-((2*inter+1e-5)/(union+1e-5)).mean(); loss=ce(p,y)+dl
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tl+=loss.item()
        model.eval(); dices=np.zeros(4)
        with torch.no_grad():
            for x,y,_ in tqdm(vdl,desc=f"{run} ep{ep+1} val",leave=False):
                x,y=x.to(device),y.to(device)
                with torch.amp.autocast('cuda'): p=model(x)
                pc=p.argmax(1)
                for c in range(4):
                    pm=(pc==c).float(); tm=(y==c).float(); tp=(pm*tm).sum(); fp=(pm*(1-tm)).sum(); fn=((1-pm)*tm).sum()
                    dices[c]+=((2*tp+1e-5)/(2*tp+fp+fn+1e-5)).item()
        dices/=len(vdl); md_=dices[1:].mean(); dur=time.time()-t0; mem=torch.cuda.max_memory_allocated()/1e9
        print(f"   {run} ep{ep+1}/{epochs}: dice={md_:.4f} ({dur:.0f}s,{mem:.1f}GB)")
        hist.append([ep+1, tl/len(tdl), md_, *dices, dur, mem])
        torch.save({'epoch':ep+1,'model':model.state_dict(),'opt':opt.state_dict(),'scheduler':sch.state_dict(),'history':hist},ckpt)
        if md_>=max(h[2] for h in hist): torch.save(model.state_dict(),os.path.join(sd,"best_model.pth"))
        pd.DataFrame(hist,columns=['epoch','train_loss','mean_dice','d_bg','d_ncr','d_ed','d_et','time','mem']).to_csv(os.path.join(sd,"log.csv"),index=False)
        sch.step()
    print(f"   finished {run}")
print("train_one_fold ready.")

In [ ]:
# ============================================================
#  RUN ALL 20 TRAININGS (4 activations x 5 folds), resumable.
# ============================================================
import sys
def _quiet(u):
    e=u.exc_value
    if isinstance(e,AssertionError) and "child process" in str(e): return
    sys.__unraisablehook__(u)
sys.unraisablehook=_quiet

for fold,(tr,va) in enumerate(FOLDS):
    train_files=[all_imgs[i] for i in tr]; val_files=[all_imgs[i] for i in va]
    for act in ACTIVATIONS:
        print(f"\n{'='*60}\n  {act.upper()}  |  fold {fold}\n{'='*60}")
        train_one_fold(act, fold, train_files, val_files, EPOCHS)
print("\nAll CV trainings complete (or resumed).")

## 5 · Evaluation — out-of-fold predictions
Each patient is scored by the model trained on the other folds. Writes `raw_predictions_cv.csv`.

In [ ]:
from scipy.ndimage import distance_transform_edt as distance
def compute_metrics(pred,gt):
    pred=pred.astype(bool); gt=gt.astype(bool)
    if pred.sum()==0 and gt.sum()==0: return 1.0,0.0,1.0,1.0
    inter=(pred&gt).sum(); dice=(2.*inter)/(pred.sum()+gt.sum()+1e-5)
    sens=inter/(gt.sum()+1e-5); prec=inter/(pred.sum()+1e-5)
    try:
        if pred.sum()>0 and gt.sum()>0:
            hd95=np.percentile(np.hstack([distance(1-gt)[pred],distance(1-pred)[gt]]),95)
        else: hd95=373.0
    except Exception: hd95=373.0
    return dice,hd95,sens,prec

RAW_CV=os.path.join(RESULTS_ROOT,"raw_predictions_cv.csv")
def evaluate_cv():
    done=set()
    if os.path.exists(RAW_CV):
        done=set(map(tuple, pd.read_csv(RAW_CV)[['Activation','Fold','Patient']].values.tolist()))
    for fold,(tr,va) in enumerate(FOLDS):
        val_files=[all_imgs[i] for i in va]
        vdl=DataLoader(CVDataset(val_files),batch_size=1,shuffle=False,num_workers=2)
        for act in ACTIVATIONS:
            mf=os.path.join(RESULTS_ROOT,f"{act}_fold{fold}","best_model.pth")
            if not os.path.exists(mf): print(f"   skip {act}_fold{fold}"); continue
            m=ImprovedUNet3D(4,4,act).to(device); m.load_state_dict(torch.load(mf,map_location=device)); m.eval()
            print(f"eval {act}_fold{fold}"); buf=[]
            with torch.no_grad():
                for x,y,fn in tqdm(vdl,leave=False):
                    pid=fn[0]
                    if (act,fold,pid) in done: continue
                    x=x.to(device); yn=y.numpy()[0]
                    with torch.amp.autocast('cuda'): pc=m(x).argmax(1).cpu().numpy()[0]
                    reg={'WT':(pc>0,yn>0),'TC':((pc==1)|(pc==3),(yn==1)|(yn==3)),'ET':(pc==3,yn==3),
                         'NCR':(pc==1,yn==1),'ED':(pc==2,yn==2)}
                    row={'Activation':act,'Fold':fold,'Patient':pid}
                    for rn,(pm,tm) in reg.items():
                        d,h,s,p=compute_metrics(pm.astype(np.uint8),tm.astype(np.uint8))
                        row[f'Dice_{rn}']=d; row[f'HD95_{rn}']=h; row[f'Sens_{rn}']=s; row[f'Prec_{rn}']=p
                    buf.append(row)
                    if len(buf)>=10:
                        pd.DataFrame(buf).to_csv(RAW_CV,mode='a',header=not os.path.exists(RAW_CV),index=False); buf=[]
            if buf: pd.DataFrame(buf).to_csv(RAW_CV,mode='a',header=not os.path.exists(RAW_CV),index=False)
    print("saved",RAW_CV)
evaluate_cv()

## 6 · Aggregate across folds
`across_fold_summary.csv` = mean ± SD over the 5 folds (Table 12 of the paper). Pooled out-of-fold = all 369 patients, each once.

In [ ]:
import math
from scipy.stats import wilcoxon
df=pd.read_csv(RAW_CV)
REG=['ET','TC','WT','NCR','ED']; MET=['Dice','HD95','Sens','Prec']
cols=[f'{m}_{r}' for m in MET for r in REG]
per_fold=df.groupby(['Activation','Fold'])[cols].mean().reset_index()
rows=[]
for a in ACTIVATIONS:
    s=per_fold[per_fold.Activation==a]; r={'Activation':a,'n_folds':s['Fold'].nunique()}
    for c in cols: r[f'{c}_mean']=s[c].mean(); r[f'{c}_sd']=s[c].std(ddof=1)
    rows.append(r)
across=pd.DataFrame(rows); across.to_csv(os.path.join(RESULTS_ROOT,"across_fold_summary.csv"),index=False)
per_fold.to_csv(os.path.join(RESULTS_ROOT,"per_fold_summary.csv"),index=False)

print("="*70); print("  5-FOLD CROSS-VALIDATION — Dice (mean ± SD over folds)"); print("="*70)
print(f"{'Act':8} | "+" | ".join(f"{r:^14}" for r in REG))
for a in ACTIVATIONS:
    rr=across[across.Activation==a].iloc[0]
    print(f"{a:8} | "+" | ".join(f"{rr[f'Dice_{r}_mean']:.3f}±{rr[f'Dice_{r}_sd']:.3f}" for r in REG))

# pooled out-of-fold Swish vs ReLU, paired by patient
print("\n  Pooled out-of-fold (369 patients), Swish vs ReLU:")
for r in ['NCR','ET']:
    mg=pd.merge(df[df.Activation=='swish'][['Patient',f'Dice_{r}']],
                df[df.Activation=='relu'][['Patient',f'Dice_{r}']],on='Patient',suffixes=('','_b'))
    try: _,p=wilcoxon(mg[f'Dice_{r}'],mg[f'Dice_{r}_b'])
    except Exception: p=float('nan')
    sw=mg[f'Dice_{r}'].mean(); rl=mg[f'Dice_{r}_b'].mean()
    print(f"   {r}: Swish {sw:.3f} vs ReLU {rl:.3f}  (Δ={ (sw-rl)*100:+.2f} pts)  p={p:.4g}")
print("\nSaved: across_fold_summary.csv, per_fold_summary.csv, raw_predictions_cv.csv")

## 7 · Outputs
The three CSV files written to `BraTS_CV_Results/` are archived in this repository under `results/crossval/`.